In [0]:
%run /Workspace/Users/nk1956663@gmail.com/Functions

In [0]:
connect()

In [0]:
# Reading Products file 

silver_path_products  = "/Volumes/proj_databricks/retail/silver/Products_silver"


products_df  = spark.read \
    .format("delta") \
    .load(silver_path_products )

display(products_df)


In [0]:
# Reading Customer file 

silver_path_customers = "/Volumes/proj_databricks/retail/silver/Customer_silver"


customers_df  = spark.read \
    .format("delta") \
    .load(silver_path_customers)

display(customers_df )


In [0]:
# Reading Orders File
# Path for Orders File

silver_path_orders = "/Volumes/proj_databricks/retail/silver/Orders_silver"

orders_df  = spark.read \
    .format("delta") \
    .load(silver_path_orders)

display(orders_df )


In [0]:
# Create DimCustomer 


dim_customer = customers_df.select(
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "phone",
    "customer_type",
    "age",
    "city",
    "state",
    "country",
    "signup_date"
).dropDuplicates(["customer_id"])

display(dim_customer)

In [0]:
# Save DimCustomer In Gold


gold_path_dim_customer = "/Volumes/proj_databricks/retail/gold/DimCustomer"

dim_customer.write \
.format("delta") \
.mode("append") \
.save(gold_path_dim_customer)

In [0]:
# Create DimProducts


dim_product = products_df.select(
    "product_id",
    "product_name",
    "category",
    "subcategory",
    "brand",
    "supplier",
    "cost_price",
    "selling_price",
    "stock_quantity",
    "rating",
    "launch_date",
    "is_active"
).dropDuplicates(["product_id"])

display(dim_product)

In [0]:
# save DimProducts in Gold


gold_path_dim_products = "/Volumes/proj_databricks/retail/gold/DimProducts"

dim_product.write \
.format("delta") \
.mode("append") \
.save(gold_path_dim_products)


In [0]:
# Create DimDate

from pyspark.sql.functions import (
    col,
    date_format,
    dayofmonth,
    month,
    quarter,
    year,
    weekofyear
)

dim_date = (
    orders_df
    .select("order_date")
    .withColumn("date_key", date_format(col("order_date"), "yyyyMMdd").cast("int"))
    .withColumn("full_date", col("order_date"))
    .withColumn("day", dayofmonth(col("order_date")))
    .withColumn("month", month(col("order_date")))
    .withColumn("month_name", date_format(col("order_date"), "MMMM"))
    .withColumn("quarter", quarter(col("order_date")))
    .withColumn("year", year(col("order_date")))
    .withColumn("week", weekofyear(col("order_date")))
    .withColumn("weekday", date_format(col("order_date"), "EEEE"))
    .select(
        "date_key",
        "full_date",
        "day",
        "month",
        "month_name",
        "quarter",
        "year",
        "week",
        "weekday"
    )
    .dropDuplicates(["date_key"])
)

display(dim_date)

In [0]:
# Save DimDate in Gold


gold_path_dim_date = "/Volumes/proj_databricks/retail/gold/DimDate"

dim_date.write \
.format("delta") \
.mode("append") \
.save(gold_path_dim_date)

In [0]:
# Create FactOrders

from pyspark.sql.functions import col, date_format

fact_orders = orders_df.withColumn(
    "total_sales",
    (
        (col("quantity") * col("unit_price"))
        - col("discount_amount")
        + col("tax_amount")
        + col("shipping_cost")
    )
).withColumn(
    "order_date_key",
    date_format(col("order_date"), "yyyyMMdd").cast("int")
).withColumn(
    "delivery_date_key",
    date_format(col("delivery_date"), "yyyyMMdd").cast("int")
).select(
    "order_id",
    "customer_id",
    "product_id",
    "order_date_key",
    "delivery_date_key",
    "warehouse_id",
    "payment_method",
    "shipment_mode",
    "city",
    "state",
    "country",
    "quantity",
    "unit_price",
    "discount_percent",
    "discount_amount",
    "tax_amount",
    "shipping_cost",
    "total_sales",
    "order_status"
)

display(fact_orders)

In [0]:
# save fact orders in Gold

gold_path_fact_orders = "/Volumes/proj_databricks/retail/gold/fact_orders"

fact_orders.write \
.format("delta") \
.mode("append") \
.save(gold_path_fact_orders)

In [0]:
# 1. Fact Orders
spark.read.format("delta").load("/Volumes/proj_databricks/retail/gold/fact_orders") \
    .write.format("delta").mode("overwrite").saveAsTable("proj_databricks.retail.fact_orders")

# 2. Dim Customer
spark.read.format("delta").load("/Volumes/proj_databricks/retail/gold/DimCustomer") \
    .write.format("delta").mode("overwrite").saveAsTable("proj_databricks.retail.dim_customer")

# 3. Dim Product
spark.read.format("delta").load("/Volumes/proj_databricks/retail/gold/DimProducts") \
    .write.format("delta").mode("overwrite").saveAsTable("proj_databricks.retail.dim_product")

# 4. Dim Date
spark.read.format("delta").load("/Volumes/proj_databricks/retail/gold/DimDate") \
    .write.format("delta").mode("overwrite").saveAsTable("proj_databricks.retail.dim_date")